# 05 — Machine Learning

**Input:** `data/processed/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`

**Goals:**
- Establish a naive baseline
- Fit and evaluate: Linear Regression, Random Forest, Gradient Boosting (XGBoost / LightGBM)
- Use time-series-aware cross-validation
- Analyse feature importance
- Tune one model with grid/random search
- Final comparison table

**Target:** `cnt` (total hourly bike rentals) — **regression** task

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb
import sys
sys.path.append("..")
from src.utils import regression_metrics
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load data

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test  = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test  = pd.read_csv("../data/processed/y_test.csv").squeeze()

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
X_train.head(3)

## 2. Baseline — predicting the mean

In [ ]:
baseline_pred = np.full(len(y_test), y_train.mean())
regression_metrics(y_test, baseline_pred, "Mean baseline")

## 3. Linear Regression & Ridge

In [ ]:
# Linear regression needs scaling — wrap in a pipeline
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])
pipe_lr.fit(X_train, y_train)
regression_metrics(y_test, pipe_lr.predict(X_test), "Linear Regression")

In [ ]:
# TODO: try Ridge with a few alpha values — which helps most?
# YOUR CODE HERE

## 4. Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
regression_metrics(y_test, rf.predict(X_test), "Random Forest")

In [ ]:
# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
feat_imp.head(15).plot(kind="bar", figsize=(12, 4), title="RF Feature Importance")
plt.tight_layout()

## 5. XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
regression_metrics(y_test, xgb_model.predict(X_test), "XGBoost")

## 6. LightGBM

In [ ]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(X_train, y_train)
regression_metrics(y_test, lgb_model.predict(X_test), "LightGBM")

## 7. Time-series cross-validation

In [ ]:
# TimeSeriesSplit respects temporal order — no data leakage
tscv = TimeSeriesSplit(n_splits=5)

for name, model in [("RF", rf), ("XGB", xgb_model), ("LGB", lgb_model)]:
    scores = cross_val_score(model, X_train, y_train, cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    print(f"{name}: RMSE = {-scores.mean():.1f} ± {scores.std():.1f}")

## 8. Hyperparameter tuning (LightGBM)

In [ ]:
# TODO: use RandomizedSearchCV or Optuna to tune LightGBM
# Key parameters: num_leaves, learning_rate, n_estimators, min_child_samples, reg_alpha, reg_lambda
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "num_leaves": [31, 63, 127],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [200, 500],
    "min_child_samples": [20, 50, 100],
}

# YOUR CODE HERE — RandomizedSearchCV with tscv

## 9. Actual vs Predicted — best model

In [ ]:
# TODO: pick your best model, plot actual vs predicted
best_pred = lgb_model.predict(X_test)

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_pred, alpha=0.3, s=10)
plt.plot([0, y_test.max()], [0, y_test.max()], "r--", lw=1)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted — LightGBM")
plt.tight_layout()

## 10. Reflect

- What's the best model and why?
- Where does the model struggle most? (peek at residuals by hour / season)
- What additional features could help?
- How would you handle this as a production pipeline?